# Phase 9: model-scale comparison
This notebook evaluates **validation AUPRC only** on the immutable `gene-split-v1` dataset. It contains no test-evaluation command. Run one model cell at a time and record its MLflow run ID and committed JSON report before continuing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi
%pip install -q transformers==4.42.3 tokenizers==0.19.1 huggingface-hub==0.23.4 mlflow==2.14.1 pandas==2.2.2 scikit-learn==1.5.0
!git clone https://github.com/djasleen15/genomic-variant-prioritizer.git /content/genomic-variant-prioritizer || git -C /content/genomic-variant-prioritizer pull --ff-only
%cd /content/genomic-variant-prioritizer
from pathlib import Path
dataset = Path('/content/drive/MyDrive/variantfx/data/labeled_split_dataset.tsv')
root = Path('/content/drive/MyDrive/variantfx/phase9')
mlruns = root / 'mlruns'
assert dataset.exists(), f'Missing fixed dataset: {dataset}'
mlruns.mkdir(parents=True, exist_ok=True)

## 50M
Completed validation result can be retained as the 50M candidate. Rerun only if an explicit clean replication is desired.

In [ ]:
!python -m src.models.finetune_lm --input "{dataset}" --output-dir "{root / 'model_scale/50m'}" --mlflow-dir "{mlruns}" --model-name InstaDeepAI/nucleotide-transformer-v2-50m-multi-species --batch-size 16 --learning-rate 2e-5 --weight-decay 0.01 --epochs 3 --patience 1 --pooling mutation --validation-only --experiment-name phase9-model-scale

## 100M — run after reviewing the 50M validation result

In [ ]:
!python -m src.models.finetune_lm --input "{dataset}" --output-dir "{root / 'model_scale/100m'}" --mlflow-dir "{mlruns}" --model-name InstaDeepAI/nucleotide-transformer-v2-100m-multi-species --batch-size 8 --learning-rate 2e-5 --weight-decay 0.01 --epochs 3 --patience 1 --pooling mutation --validation-only --experiment-name phase9-model-scale

The optional 250M run is deliberately omitted until 100M memory usage is reviewed. Do not evaluate the test split from this notebook.